# 5. Basic Structure Operations

In [1]:
import os
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-23.jdk/Contents/Home"  # or wherever your JDK 17/21 lives
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [2]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder.appName("Local Spark Session")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "spark-warehouse")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .enableHiveSupport()
    .getOrCreate()
)
print(spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/21 07:54:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://127.0.0.1:4040


In [7]:
df = spark.read.format("json").load("../data/flight-data/json/2015-summary.json")
df.printSchema()
df.schema

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: long (nullable = true)



StructType([StructField('DEST_COUNTRY_NAME', StringType(), True), StructField('ORIGIN_COUNTRY_NAME', StringType(), True), StructField('count', LongType(), True)])

## To retain

**Schema on read - We let the data source define the schema**

**The manipulation of columns are called expressions**

**Row objects in Pyspark DF represent internally an array of bytes**

* Columns are just expressions;
* Columns and transformations of those columns compile to the same logical plan as parsed expressions;


Example:
(((col("someCol") + 5) * 200) - 6) < col("otherCol")

![alt text](image.png)

This graph is familiar with the acyclic graph

In [30]:
# How to enforce a schema on a DataFrame

from pyspark.sql.types import StructField, StructType, StringType, LongType
import pyspark.sql.functions as F

myManualSchema = StructType([
    StructField("DEST_COUNTRY_NAME", StringType(), True),
    StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
    StructField("count", LongType(), False, metadata={"hello": "world"})
])

df = spark.read.format("json").schema(myManualSchema).load("../data/flight-data/json/2015-summary.json")

F.col("someColumnName")
F.column("someColumnName")


print(df.columns)
df.count()

print(df.first())


# we can create a row
from pyspark.sql import Row
myRow = Row("Hello", None, 1, False)
myRow[0]

['DEST_COUNTRY_NAME', 'ORIGIN_COUNTRY_NAME', 'count']
Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Romania', count=15)


'Hello'

In [35]:
df = spark.read.format("json").load("../data/flight-data/json/2015-summary.json")
df.createOrReplaceTempView("dfTable")


from pyspark.sql import Row
import pyspark.sql.types as T

myManualSchema = T.StructType([
    T.StructField("some", T.StringType(), True),
    T.StructField("col", T.StringType(), True),
    T.StructField("names", T.LongType(), False)
])

myRow = Row("Hello", None, 1)
myDf = spark.createDataFrame([myRow], myManualSchema)
myDf.show()

+-----+----+-----+
| some| col|names|
+-----+----+-----+
|Hello|NULL|    1|
+-----+----+-----+



In [41]:
# Select ant SelectExpr in Dataframes
df.select("DEST_COUNTRY_NAME").show(2)
df.select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME").show(2)


df.select(
    F.expr("DEST_COUNTRY_NAME"),
    F.col("DEST_COUNTRY_NAME"),
    F.column("DEST_COUNTRY_NAME")
).show(2)


df.select(F.expr("DEST_COUNTRY_NAME AS destination")).show(2)

# We can manipulate the result of an expression as another expression
df.select(F.expr("DEST_COUNTRY_NAME AS destination")).alias("DEST_COUNTRY_NAME").show(2)

# Since EXPR is so used, Spark created the Shortcut, selectExpr

df.selectExpr("DEST_COUNTRY_NAME as newColumnName", "DEST_COUNTRY_NAME").show(2)

+-----------------+
|DEST_COUNTRY_NAME|
+-----------------+
|    United States|
|    United States|
+-----------------+
only showing top 2 rows
+-----------------+-------------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|
+-----------------+-------------------+
|    United States|            Romania|
|    United States|            Croatia|
+-----------------+-------------------+
only showing top 2 rows
+-----------------+-----------------+-----------------+
|DEST_COUNTRY_NAME|DEST_COUNTRY_NAME|DEST_COUNTRY_NAME|
+-----------------+-----------------+-----------------+
|    United States|    United States|    United States|
|    United States|    United States|    United States|
+-----------------+-----------------+-----------------+
only showing top 2 rows
+-------------+
|  destination|
+-------------+
|United States|
|United States|
+-------------+
only showing top 2 rows
+-------------+
|  destination|
+-------------+
|United States|
|United States|
+-------------+
only showing top

In [44]:
# More advanced logic, how to create a new column with selectExpr
df.selectExpr(
    "*", # get all origin columns
    "(DEST_COUNTRY_NAME = ORIGIN_COUNTRY_NAME) as withinCountry"
).show(2)

# We can also use aggregations
df.selectExpr(
    "avg(count)", "count(distinct(DEST_COUNTRY_NAME))"
).show(2)


# Literals allows us to pass a value from a programming language into a spark value that he can understand
df.select(F.expr("*"), F.lit(1).alias("One")).show(2)

+-----------------+-------------------+-----+-------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|withinCountry|
+-----------------+-------------------+-----+-------------+
|    United States|            Romania|   15|        false|
|    United States|            Croatia|    1|        false|
+-----------------+-------------------+-----+-------------+
only showing top 2 rows
+-----------+---------------------------------+
| avg(count)|count(DISTINCT DEST_COUNTRY_NAME)|
+-----------+---------------------------------+
|1770.765625|                              132|
+-----------+---------------------------------+

+-----------------+-------------------+-----+---+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|One|
+-----------------+-------------------+-----+---+
|    United States|            Romania|   15|  1|
|    United States|            Croatia|    1|  1|
+-----------------+-------------------+-----+---+
only showing top 2 rows


In [ ]:
# we can make spark to be Case Sensitive
spark.conf.set("spark.sql.caseSensitive", "false")

# Adding columns
df.withColumn("numberOne", F.lit(1)).show(2)

df.withColumn("withinCountry", F.expr("ORIGIN_COUNTRY_NAME == DEST_COUNTRY_NAME")).show(2)

# how to rename columns
df.withColumnRenamed("DEST_COUNTRY_NAME", "dest").columns


# We can create columns with Spaces on it
# On this case, there is not need to use ` because the string parameter accepts on the WithColumn
df_with_big_column = df.withColumn("This Long Column-name", F.col("ORIGIN_COUNTRY_NAME"))

df_with_big_column.selectExpr(
    "`This long Column-name`",
    "`This long Column-name` as `new col`"
 ).show(2)


# Cast type columns
df.withColumn("count2", F.col("count").cast("long"))

+-----------------+-------------------+-----+---------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|numberOne|
+-----------------+-------------------+-----+---------+
|    United States|            Romania|   15|        1|
|    United States|            Croatia|    1|        1|
+-----------------+-------------------+-----+---------+
only showing top 2 rows
+-----------------+-------------------+-----+-------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|withinCountry|
+-----------------+-------------------+-----+-------------+
|    United States|            Romania|   15|        false|
|    United States|            Croatia|    1|        false|
+-----------------+-------------------+-----+-------------+
only showing top 2 rows
+---------------------+-------+
|This long Column-name|new col|
+---------------------+-------+
|              Romania|Romania|
|              Croatia|Croatia|
+---------------------+-------+
only showing top 2 rows


DataFrame[DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string, count: bigint, count2: bigint]

In [58]:
# Filtering rows
# To filter rows, we create an expression that evaluates to True or False

df.filter(F.col("count") < 2).show(2)
df.where("count < 2").show(2)


# Getting unique rows
df.select("ORIGIN_COUNTRY_NAME").distinct().count()

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Croatia|    1|
|    United States|          Singapore|    1|
+-----------------+-------------------+-----+
only showing top 2 rows
+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Croatia|    1|
|    United States|          Singapore|    1|
+-----------------+-------------------+-----+
only showing top 2 rows


125

In [59]:
# Random samples
seed = 5
withReplacement = False
fraction = 0.5
df.sample(withReplacement, fraction, seed).count()

138